# 🧠 Fine-Tuning Transformers for Question Answering (SQuAD v1.1)

> **Notebook generated:** 2025-09-01 18:24 UTC

> **Assignment Goals**  
> 1) Learn how QA differs from classification tasks  
> 2) Fine-tune a BERT-based model on **SQuAD v1.1** using Hugging Face  
> 3) Evaluate with **Exact Match (EM)** and **F1**  
> 4) Test on real questions  
> 5) Share your Colab notebook publicly

---

## ✅ How QA differs from Classification
**Text classification** predicts a single label for the whole input text (e.g., positive/negative).  
**Extractive Question Answering (QA)**, however, predicts **a span of text** (start and end positions) **within a context** that answers a given question.

Key differences:
- **Inputs**: QA takes a **(question, context)** pair; classification takes a single text.
- **Outputs**: QA returns **text spans** (start/end token positions) rather than a single label.
- **Preprocessing**: QA must **align character-level answer spans** to **token positions** after tokenization.
- **Evaluation**: QA uses **Exact Match (EM)** and **F1** on text answers, not accuracy on labels.

## 🔧 1) Setup (Colab)
Run the cell below to install the required libraries.

In [ ]:
!pip -q install transformers datasets evaluate accelerate

## 📚 2) Load & Explore SQuAD v1.1
We will use the Hugging Face `datasets` library to load **SQuAD v1.1**.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("squad")
dataset

### Peek at a few samples
Each example has: **id, title, context, question, answers**

In [ ]:
sample = dataset["train"][0]
for k, v in sample.items():
    print(f"{k}: {v if k != 'context' else v[:300] + '...'}")

## ✂️ 3) Tokenization & Answer Span Mapping
We'll use the **BERT base uncased** tokenizer.  
We must **map character-level answer spans** to **token start/end positions** in the tokenized sequence, handling long contexts with sliding window (doc stride).

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

# Hyperparameters for max length + stride (standard choices for SQuAD-style QA)
max_length = 384
doc_stride = 128

pad_on_right = tokenizer.padding_side == "right"

def prepare_train_features(examples):
    # Tokenize our examples with truncation and maybe padding, but keep the overflows using a stride.
    # This results in one example possible giving several features when a context is long.
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    # Let's label those examples!
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # Grab the sequence corresponding to that example
        sequence_ids = tokenized_examples.sequence_ids(i)

        # One example can give several spans, this is the index of the example containing this span.
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        # If no answers are given, set the cls_index as answer.
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Start/end character index of the answer in the text.
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Start token index of the current span in the text.
        token_start_index = 0
        while sequence_ids[token_start_index] != (1 if pad_on_right else 0):
            token_start_index += 1

        # End token index of the current span in the text.
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != (1 if pad_on_right else 0):
            token_end_index -= 1

        # If the answer is not fully inside the span, label it (cls_index).
        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            # Otherwise move the token_start_index and token_end_index to the two ends of the answer.
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    return tokenized_examples

def prepare_validation_features(examples):
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    tokenized_examples["example_id"] = []

    # For validation, we need to keep the example_id and an offset mapping.
    for i in range(len(tokenized_examples["input_ids"])):
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1 if pad_on_right else 0

        sample_index = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_index])

        # Set to None the offset_mapping that are not part of the context so they are ignored at post-processing.
        offsets = tokenized_examples["offset_mapping"][i]
        tokenized_examples["offset_mapping"][i] = [
            (o if sequence_ids[k] == context_index else None) for k, o in enumerate(offsets)
        ]

    return tokenized_examples

### Apply preprocessing
Map the dataset with our preprocessing functions (this can take a few minutes in Colab).

In [ ]:
# Set num_proc>1 in Colab to speed up mapping if desired.
train_dataset = dataset["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

validation_dataset = dataset["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=dataset["validation"].column_names,
)
len(train_dataset), len(validation_dataset)

## 🧩 4) Model Setup
Load **BERT base uncased** for question answering.

In [ ]:
from transformers import AutoModelForQuestionAnswering, DataCollatorWithPadding

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 🚀 5) Fine-tuning with `Trainer`
We train for **2–3 epochs**, batch size **16**, learning rate **3e-5**.

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
import torch

batch_size = 16

training_args = TrainingArguments(
    output_dir="./qa-bert-squad",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

## 📏 6) Evaluation — Exact Match (EM) & F1
We will **post-process** model logits to text spans and compute **EM** and **F1** using `evaluate.load("squad")`.

In [ ]:
import collections
import evaluate
import numpy as np
from tqdm.auto import tqdm

metric = evaluate.load("squad")

def postprocess_qa_predictions(examples, features, predictions, n_best_size=20, max_answer_length=30):
    # Based on HF QA evaluation recipe
    all_start_logits, all_end_logits = predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = collections.OrderedDict()

    print(f"Post-processing {len(examples)} example predictions split into {len(features)} features.")

    for example_index, example in enumerate(tqdm(examples)):
        feature_indices = features_per_example[example_index]
        min_null_score = None  # for possible future squad v2
        valid_answers = []

        context = example["context"] if "context" in example else ""

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]
            # Optional: ignore cls token (non-context) positions
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    if end_index < start_index:
                        continue
                    length = end_index - start_index + 1
                    if length > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = context[start_char: end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": text})

        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            best_answer = {"text": "", "score": 0.0}

        predictions[example["id"]] = best_answer["text"]

    return predictions

# Run prediction on the validation features
raw_predictions = trainer.predict(validation_dataset)
start_logits, end_logits = raw_predictions.predictions

# We need the original validation examples to build references
validation_examples = dataset["validation"]

# Build processed predictions dict: id -> text
final_predictions = postprocess_qa_predictions(
    validation_examples,
    validation_dataset,
    (start_logits, end_logits),
)

# Build references list for metric
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in validation_examples]

metrics = metric.compute(predictions=[{"id": k, "prediction_text": v} for k, v in final_predictions.items()],
                         references=references)
metrics

## 🧪 7) Real-World Q&A Tests
Try a few test questions manually.

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

question_answering_model = model
qa_tokenizer = tokenizer

def answer_question(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384)
    with torch.no_grad():
        outputs = question_answering_model(**inputs)
    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    start_idx = torch.argmax(start_logits).item()
    end_idx = torch.argmax(end_logits).item()

    all_tokens = qa_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    answer = qa_tokenizer.convert_tokens_to_string(all_tokens[start_idx: end_idx + 1])
    return answer

# Sample 1 (provided in assignment)
q1 = "Who developed the theory of relativity?"
c1 = "Albert Einstein developed the theory of relativity in the early 20th century."
print("Q1:", q1)
print("A1:", answer_question(q1, c1))

# Custom Test 1
q2 = "What is the capital of France?"
c2 = "Paris is the capital and most populous city of France, situated on the River Seine."
print("\nQ2:", q2)
print("A2:", answer_question(q2, c2))

# Custom Test 2
q3 = "Who wrote the play Hamlet?"
c3 = "William Shakespeare wrote the tragedy Hamlet, which is one of his most famous works."
print("\nQ3:", q3)
print("A3:", answer_question(q3, c3))

## 💾 Save Model & Tokenizer

In [ ]:
save_path = "./bert-squad-finetuned"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
save_path

## 🌐 Share your Colab Notebook
1. Go to **File → Save a copy in Drive** (if needed).  
2. Then **Share** → set to **Anyone with the link** (Viewer).  
3. Copy the link and submit it as required.

## ✨ Reflection (5–6 sentences)
- In this assignment, I learned how extractive QA differs from text classification, especially the need to map character-level answers to token spans.  
- I practiced sliding windows with `doc_stride` to handle long contexts and how to post-process logits into readable answers.  
- I explored evaluation using **Exact Match** and **F1**, which compare predicted answer strings to ground-truth spans.  
- Training with Hugging Face `Trainer` simplified the pipeline from data to model to evaluation.  
- If I extended this work, I would experiment with better pre-trained checkpoints (e.g., `distilbert-base-uncased`, `deberta-base`), larger `max_length`, and more robust decoding (n-best lists).  
- I also learned about the importance of **tokenizer alignment** and why span indices can become tricky after subword tokenization.